In [1]:
import json
import time

import requests
from kafka import KafkaConsumer, KafkaProducer
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

producer = KafkaProducer(bootstrap_servers='kafka:9092')
consumer = KafkaConsumer(bootstrap_servers="kafka:9092")

producer.send('weather_stream', b'Bonjour Kafka')
producer.flush()

print(consumer.topics())
consumer.close()

{'weather_stream'}


In [2]:
consumer = KafkaConsumer(
    "weather_stream",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda value: value.decode("utf-8"),
    consumer_timeout_ms=5000,  # s'arrête après 5 secondes sans nouveau message
)

for message in consumer:
    print(message.value)

consumer.close()

Bonjour Kafka


In [ ]:
cities = [
    {"city": "Paris", "country": "France", "latitude": 48.8566, "longitude": 2.3522},
    {"city": "Seville", "country": "Spain", "latitude": 37.3891, "longitude": -5.9845},
    {"city": "Dubai", "country": "United Arab Emirates", "latitude": 25.2048, "longitude": 55.2708},
    {"city": "Reykjavik", "country": "Iceland", "latitude": 64.1466, "longitude": -21.9426},
    {"city": "Wellington", "country": "New Zealand", "latitude": -41.2865, "longitude": 174.7762},
]

weather_producer = KafkaProducer(bootstrap_servers="kafka:9092")
api_url = "https://api.open-meteo.com/v1/forecast"
interval_seconds = 30

print("Envoi météo démarré. Interromps la cellule pour arrêter.")
try:
    while True:
        cycle_start = time.monotonic()
        for city_info in cities:
            try:
                response = requests.get(
                    api_url,
                    params={
                        "latitude": city_info["latitude"],
                        "longitude": city_info["longitude"],
                        "current": "temperature_2m,wind_speed_10m,weather_code",
                        "wind_speed_unit": "ms",
                    },
                    timeout=10,
                )
                response.raise_for_status()
                weather_data = response.json()
                weather_data.update(
                    city=city_info["city"],
                    country=city_info["country"],
                )
                weather_producer.send(
                    "weather_stream",
                    json.dumps(weather_data).encode("utf-8"),
                )
                print(json.dumps(weather_data, ensure_ascii=False))
            except requests.RequestException as error:
                print(f"Erreur météo pour {city_info['city']}: {error}")

        weather_producer.flush()
        elapsed = time.monotonic() - cycle_start
        time.sleep(max(0, interval_seconds - elapsed))
except KeyboardInterrupt:
    print("Envoi météo arrêté.")
finally:
    weather_producer.flush()
    weather_producer.close()

Envoi météo démarré. Interromps la cellule pour arrêter.
{"latitude": 48.853382, "longitude": 2.355208, "generationtime_ms": 0.04756450653076172, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 36.0, "current_units": {"time": "iso8601", "interval": "seconds", "temperature_2m": "°C", "wind_speed_10m": "m/s", "weather_code": "wmo code"}, "current": {"time": "2026-11-10T16:15", "interval": 900, "temperature_2m": 12.7, "wind_speed_10m": 1.3, "weather_code": 61}, "city": "Paris", "country": "France"}
{"latitude": 37.364082, "longitude": -5.956622, "generationtime_ms": 0.11479854583740234, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 18.0, "current_units": {"time": "iso8601", "interval": "seconds", "temperature_2m": "°C", "wind_speed_10m": "m/s", "weather_code": "wmo code"}, "current": {"time": "2026-11-10T16:15", "interval": 900, "temperature_2m": 20.2, "wind_speed_10m": 4.74, "weather_code": 80}, "city": "

In [ ]:
spark = (
    SparkSession.builder
    .appName("WeatherAlerts")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

current_schema = StructType([
    StructField("temperature_2m", DoubleType()),
    StructField("wind_speed_10m", DoubleType()),
    StructField("weather_code", IntegerType()),
])
weather_schema = StructType([
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("current", current_schema),
])

kafka_source = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather_stream")
    .option("startingOffsets", "latest")
    .load()
)

parsed_weather = kafka_source.select(
    F.col("timestamp").alias("event_time"),
    F.from_json(F.col("value").cast("string"), weather_schema).alias("weather"),
).where(F.col("weather").isNotNull())

transformed_weather = parsed_weather.select(
    "event_time",
    F.col("weather.current.temperature_2m").alias("temperature"),
    F.col("weather.current.wind_speed_10m").alias("windspeed"),
    F.col("weather.city").alias("city"),
    F.col("weather.country").alias("country"),
    F.col("weather.current.weather_code").alias("weather_code"),
).withColumn(
    "wind_alert_level",
    F.when(F.col("windspeed") < 10, "level_0")
    .when(F.col("windspeed") <= 20, "level_1")
    .otherwise("level_2"),
).withColumn(
    "heat_alert_level",
    F.when(F.col("temperature") < 25, "level_0")
    .when(F.col("temperature") <= 35, "level_1")
    .otherwise("level_2"),
)

kafka_output = transformed_weather.select(
    F.to_json(F.struct(
        "event_time",
        "temperature",
        "windspeed",
        "city",
        "country",
        "weather_code",
        "wind_alert_level",
        "heat_alert_level",
    )).alias("value")
)

weather_query = (
    kafka_output.writeStream
    .format("kafka")
    .outputMode("append")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "weather_transformed")
    .option("checkpointLocation", "/home/jovyan/work/checkpoints/weather_transformed")
    .start()
)

print("Streaming Spark démarré vers le topic weather_transformed.")

In [ ]:
transformed_schema = StructType([
    StructField("event_time", TimestampType()),
    StructField("temperature", DoubleType()),
    StructField("windspeed", DoubleType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("weather_code", IntegerType()),
    StructField("wind_alert_level", StringType()),
    StructField("heat_alert_level", StringType()),
])

transformed_source = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather_transformed")
    .option("startingOffsets", "latest")
    .load()
)

transformed_events = transformed_source.select(
    F.from_json(F.col("value").cast("string"), transformed_schema).alias("weather")
).where(F.col("weather").isNotNull()).select("weather.*")

windowed_events = transformed_events.withWatermark("event_time", "1 minute")

weather_aggregates = (
    windowed_events
    .groupBy(
        F.window("event_time", "1 minute", "30 seconds").alias("window"),
        "country",
        "city",
    )
    .agg(
        F.sum(F.when(F.col("wind_alert_level") == "level_1", 1).otherwise(0)).alias("wind_level_1_count"),
        F.sum(F.when(F.col("wind_alert_level") == "level_2", 1).otherwise(0)).alias("wind_level_2_count"),
        F.sum(F.when(F.col("heat_alert_level") == "level_1", 1).otherwise(0)).alias("heat_level_1_count"),
        F.sum(F.when(F.col("heat_alert_level") == "level_2", 1).otherwise(0)).alias("heat_level_2_count"),
        F.sum(
            F.when(F.col("wind_alert_level").isin("level_1", "level_2"), 1).otherwise(0)
            + F.when(F.col("heat_alert_level").isin("level_1", "level_2"), 1).otherwise(0)
        ).alias("total_alerts"),
        F.avg("temperature").alias("avg_temperature"),
        F.min("temperature").alias("min_temperature"),
        F.max("temperature").alias("max_temperature"),
    )
)

weather_aggregate_query = (
    weather_aggregates.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", "false")
    .option("checkpointLocation", "/home/jovyan/work/checkpoints/weather_aggregates")
    .start()
)

print("Agrégations météo démarrées dans la console.")